In [ ]:
import datetime
import glob
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import time
import torch
import torch.optim as optim

import icepinn


nc_files = [pathlib.Path(p) for p in sorted(glob.glob('data/V6/*.nc'))]
npz_files = [pathlib.Path(p) for p in sorted(glob.glob('data/V6/*interp.npz'))]

d = icepinn.nc.SeaIceV6([nc_files[1]])
npz = np.load(npz_files[1])


SAVEDIR = pathlib.Path('models/ModelV1/')
if not SAVEDIR.exists():
    SAVEDIR.mkdir(parents=True)
SAVEPLUG = '{datetime}-epoch{epoch_num:03d}-batch{batch_num:03d}.pth'
GLOBSTR = '*.pth'

# load sorted(glob(GLOBSTR))[-1] into network parameters
# only has effect if at least one network param file exists
LOAD = True  
TRAIN = True  # do training, possibly after loading


DEVICE = 'cuda'
DTYPE = torch.float32
BATCHSIZE = 20000
EPOCHS = 100
# approximate number of batches needed to view every point
# inside ice pack (not ocean) across 1 year of data 
BATCHES_PER_EPOCH = 365 * 200 * 200  // BATCHSIZE

C = 3
N, M = (13, 13)
HX, HY = N // 2, M // 2

ld = len(d.date)
lx = len(d.x)
ly = len(d.y)


Initialize some stuff:

In [ ]:
U0 = 1.0000e+00
L0 = 2.5000e+04
T0 = 1.0644e+03
K0 = 1.0000e+03
V0 = 5.0000e-02
F0 = 9.3949e-04

s = 24 * 60 * 60 / T0  # day in seconds
h = 25e3 / L0

n = icepinn.network.ModelV1(
    U0,
    L0,
    T0,
    K0,
    V0,
    F0,
    s = s,
    h = h
).to(DEVICE, DTYPE)


In [ ]:
losses = []
data = np.empty((BATCHSIZE, C, N, M), dtype=np.float64)
label = np.empty((BATCHSIZE,), dtype=np.float64)

loss_weights = torch.tensor(
    [
        1,
        1,
        1,
        1,
        1,
        1,
        1,
    ],
    device=DEVICE,
    dtype=DTYPE
)


In [ ]:
if LOAD and len(param_files := glob.glob(GLOBSTR)) > 0:
    n.load(param_files[-1])

if TRAIN:
    optimizer = optim.Adam(n.parameters(), lr=1e-2)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=100, factor=0.1, min_lr=1e-4)

    n.train()
    try:
        for epoch_num in range(EPOCHS):
            for batch_num in range(BATCHES_PER_EPOCH):
                if batch_num % 50 == 0:
                    f = npz_files[np.random.randint(len(npz_files))]
                    print(f'\nShuffling (changing to file {f.name})...')
                    npz = np.load(f)
                    
                    seaice_conc = npz['seaice_conc_interp'].copy()
                    
                    valid = (~npz['mask']).copy()
                    valid[:C - 1] = False
                    valid[-1:] = False
                    valid[:, :HX] = False
                    valid[:, -HX:] = False
                    valid[:, :, :HY] = False
                    valid[:, :, -HY:] = False
                    valid[np.sum(npz['mask'], axis=(1, 2)) > 90000] = False  # mask out bad days

                    n_valid = np.sum(valid)
                    it, ix, iy = np.nonzero(valid)

                    sample_prob = (
                        (seaice_conc[valid] - 0) * 
                        (1.25 - seaice_conc[valid]) *
                        (0.75 + 1 - seaice_conc[valid])
                    ) + 1e-1
                    sample_prob = np.cumsum(sample_prob) / np.sum(sample_prob)

                tb = time.perf_counter()
                for ii, jj in enumerate(np.searchsorted(sample_prob, np.random.random_sample(BATCHSIZE))):
                    data[ii] = seaice_conc[
                        it[jj] - C + 1:it[jj] + 1,
                        ix[jj] - HX:ix[jj] + HX + 1,
                        iy[jj] - HY:iy[jj] + HY + 1
                    ]
                    label[ii] = seaice_conc[it[jj] + 1, ix[jj], iy[jj]]

                tt = time.perf_counter()
                data_t = torch.as_tensor(data, dtype=DTYPE, device=DEVICE)
                label_t = torch.as_tensor(label, dtype=DTYPE, device=DEVICE)

                optimizer.zero_grad(set_to_none=True)
                _, loss_elems = n.forward(data_t, label_t)
                loss_elems = loss_elems.mean(dim=-1)
                loss = loss_elems @ loss_weights
                if torch.any(torch.isnan(loss)):
                    print(f'\nEpoch {epoch_num + 1:4} (batch {batch_num + 1:4}):\tLoss is NaN!  Skipping regression...')
                else:
                    loss.backward()
                    optimizer.step()
                    if scheduler is not None:
                        scheduler.step(loss.detach())

                losses.append(loss.item())
                tf = time.perf_counter()
                print(
                    f'Epoch {epoch_num + 1:4} (batch {batch_num + 1:4}):\t\t'
                    f'Loss: {loss.item():10e}\t\t'
                    f'Batching time: {tt - tb:.3f} seconds\t\t'
                    f'Training time: {tf - tt:.3f} seconds\t\t',
                    end='\r'
                )

        dest = SAVEPLUG.format(
            {
                'datetime': datetime.datetime.now().strftime('%Y%m%d%H%M%S'),
                'epoch_num': epoch_num,
                'batch_num': batch_num,
            }
        )
        n.save(dest)

    except KeyboardInterrupt:
        dest = SAVEPLUG.format(
            {
                'datetime': datetime.datetime.now().strftime('%Y%m%d%H%M%S'),
                'epoch_num': epoch_num,
                'batch_num': batch_num,
            }
        )
        print(f'Caught keyboard interrupt.  Saving to {dest}...')
        n.save(dest)


We can compute the solution across a range of coordinates:

In [ ]:
cmap_ice = plt.get_cmap('Blues_r')
cmap_ice.set_bad("#ff7057", alpha=0.6)

cmap_err = plt.get_cmap('bwr_r')
cmap_err.set_bad("#ff7057", alpha=0.6)

cmap_other = plt.get_cmap('viridis')
cmap_other.set_bad("#ff7057", alpha=0.6)


def plot_outputs(outputs_np: np.ndarray, truth: np.ndarray, title: str):
    fig, axes = plt.subplots(2, 3, figsize=(15, 10), dpi=300)

    ax = axes[0, 0]
    m = ax.imshow(
        outputs_np[..., 0].T,
        cmap=cmap_ice,
        vmin=0.,
        vmax=1.
    )
    plt.colorbar(m, ax=ax, label='Fractional Sea Ice Coverage\n(Forecast)')

    ax = axes[0, 1]
    m = ax.imshow(
        (truth - outputs_np[..., 0]).T,
        cmap=cmap_err,
        vmin=-1,
        vmax=1.
    )
    plt.colorbar(m, ax=ax, label='Truth minus Forecast')

    ax = axes[0, 2]
    truth_masked = truth.copy()
    truth_masked[np.isnan(outputs_np[..., 0])] = np.nan
    m = ax.imshow(
        truth_masked.T,
        cmap=cmap_ice,
        vmin=0.,
        vmax=1.
    )
    plt.colorbar(m, ax=ax, label='Fractional Sea Ice Coverage\n(Truth)')

    ax = axes[1, 0]
    m = ax.imshow(
        K0 * outputs_np[..., 1].T / 1e3 ** 2 * 60 * 60,
        cmap=cmap_other,
        vmin=-5.,
        vmax=5.,
    )
    plt.colorbar(m, ax=ax, label='Diffusivity (km$^2$ / hr)')

    ax = axes[1, 1]
    m = ax.imshow(
        V0 * np.sqrt(np.sum((outputs_np[..., 2:4]) ** 2, axis=-1)).T / 1e3 * 60 * 60,
        cmap=cmap_other,
        vmin=0.,
        vmax=1.,
    )
    plt.colorbar(m, ax=ax, label='Velocity (km / hr)')

    ax = axes[1, 2]
    m = ax.imshow(
        F0 * outputs_np[..., 4].T * 24 * 60 * 60,
        cmap=cmap_other,
        vmin=-1.,
        vmax=1.,
    )
    plt.colorbar(m, ax=ax, label='Fractional Sea Ice (per day)')

    plt.suptitle(title)

    plt.tight_layout()
    plt.show()


def infer(f: pathlib.Path, it: int):
    n.eval()

    print(f'Using file {f.name}...')
    npz = np.load(f)

    Ni, Mi = (13, 13)
    HXi, HYi = Ni // 2, Mi // 2
    lt, lx, ly = npz['seaice_conc_interp'].shape

    truth = npz['seaice_conc_interp'][it + 1, HXi:lx - HXi, HYi:ly - HYi]

    with torch.inference_mode():
        data = torch.tensor(
            npz['seaice_conc_interp'][it - 2:it + 1][None, ...],  # unsqueeze B dimension
        ).to(dtype=DTYPE, device=DEVICE)
        patch = torch.nn.functional.unfold(data, Ni).reshape((3, Ni, Mi, (lx - 2 * HXi) * (ly - 2 * HYi)))
        patch = patch.moveaxis(-1, 0)

        outputs = torch.zeros(((lx - 2 * HXi) * (ly - 2 * HYi), 5), dtype=DTYPE, device=DEVICE)
        for i in range(len(patch) // BATCHSIZE + 1):
            outputs[i * BATCHSIZE:(i + 1) * BATCHSIZE] = n.forward(patch[i * BATCHSIZE:(i + 1) * BATCHSIZE])

        patch_mask = torch.tensor(npz['mask'][it, HXi:lx - HXi, HYi:ly - HYi]).to(device=DEVICE)
        outputs[patch_mask.ravel()] = torch.nan
        outputs = outputs.reshape(((lx - 2 * HXi), (ly - 2 * HYi), 5))

        outputs_np = outputs.numpy(force=True)

    print(
        f'RMS error: {np.sqrt(np.nansum((outputs_np[..., 0] - truth)[~npz["mask"][it, HXi:lx - HXi, HYi:ly - HYi]] ** 2))}',
        f'Mean error: {np.nanmean(np.abs((outputs_np[..., 0] - truth)[~npz["mask"][it, HXi:lx - HXi, HYi:ly - HYi]] ** 2))}',
        sep='\n'
    )
    plot_outputs(
        outputs_np,
        truth,
        f'Forecast for {npz["date"][it + 1]}'
    )


def infer_longterm(f: pathlib.Path, it: int, term: int):
    import scipy.interpolate as interpolate

    n.eval()

    print(f'Using file {f.name}...')
    npz = np.load(f)

    Ni, Mi = (13, 13)
    HXi, HYi = Ni // 2, Mi // 2
    lt, lx, ly = npz['seaice_conc_interp'].shape

    truth = npz['seaice_conc_interp'][it + term, HXi:lx - HXi, HYi:ly - HYi]

    xx, yy = np.meshgrid(npz['x'], npz['y'], indexing='ij')

    with torch.inference_mode():
        mask = torch.tensor(npz['mask'][it], device=DEVICE)
        patch_mask = mask[HXi:lx - HXi, HYi:ly - HYi]
        data = torch.tensor(
            npz['seaice_conc_interp'][it - 2:it + 1][None, ...],  # unsqueeze B dimension
        ).to(dtype=DTYPE, device=DEVICE)

        for _ in range(term):
            patch = torch.nn.functional.unfold(data, Ni).reshape((3, Ni, Mi, (lx - 2 * HXi) * (ly - 2 * HYi)))
            patch = patch.moveaxis(-1, 0)

            patch_outputs = torch.full(((lx - 2 * HXi) * (ly - 2 * HYi), 5), 0, dtype=DTYPE, device=DEVICE)
            for i in range(len(patch) // BATCHSIZE + 1):
                patch_outputs[i * BATCHSIZE:(i + 1) * BATCHSIZE] = n.forward(patch[i * BATCHSIZE:(i + 1) * BATCHSIZE])

            patch_outputs[patch_mask.ravel()] = torch.nan
            patch_outputs = patch_outputs.reshape(((lx - 2 * HXi), (ly - 2 * HYi), 5))
            patch_outputs = torch.clip(patch_outputs, 0., 1.)

            # move data around
            data = torch.roll(data, -1, 1)
            data[:, 2, mask] = torch.tensor(
                interpolate.NearestNDInterpolator(  # interpolate everywhere we didn't see
                    np.stack(
                        (
                            xx[HXi:lx - HXi, HYi:ly - HYi][~npz['mask'][it, HXi:lx - HXi, HYi:ly - HYi]].ravel(),
                            yy[HXi:lx - HXi, HYi:ly - HYi][~npz['mask'][it, HXi:lx - HXi, HYi:ly - HYi]].ravel()
                        ),
                        axis=-1
                    ),
                    patch_outputs[~patch_mask, 0].numpy(force=True).ravel()
                )(xx[npz['mask'][it]], yy[npz['mask'][it]]),
                dtype=DTYPE,
                device=DEVICE
            )

        patch_outputs_np = patch_outputs.numpy(force=True)

    print(
        f'RMS error: {np.sqrt(np.nansum((patch_outputs_np[..., 0] - truth)[~npz["mask"][it, HXi:lx - HXi, HYi:ly - HYi]] ** 2))}',
        f'Mean error: {np.nanmean(np.abs((patch_outputs_np[..., 0] - truth)[~npz["mask"][it, HXi:lx - HXi, HYi:ly - HYi]] ** 2))}',
        sep='\n'
    )
    plot_outputs(
        patch_outputs_np,
        truth,
        f'{term}-day Forecast for {npz["date"][it + term]}'
    )


def infer_from_logical_names(year: int, day_of_year: int, term: int | None = None):
    if term:
        infer_longterm(npz_files[year - 1978], day_of_year - 1, term)
    else:
        infer(npz_files[year - 1978], day_of_year - 1)


In [ ]:
infer_from_logical_names(2012, 155)


In [ ]:
infer_from_logical_names(2012, 155, 14)
